# Notebook 67 v2 — Policy Learning

**Repository target:** `confidence-scheduled-decoding`

This notebook generates the figure set for Notebook 67: policy learning from validated telemetry records. It writes PNG outputs into `figures/` and creates a zip of those outputs for upload or lab-report use.


In [ ]:
from pathlib import Path
import json, zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import ListedColormap, BoundaryNorm

OUT = Path("figures")
OUT.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 180,
    "font.size": 18,
    "axes.titlesize": 34,
    "axes.labelsize": 24,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16,
})

def savefig(name):
    path = OUT / name
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    return path

def rounded_box(ax, xy, w, h, text, fontsize=18, lw=2.2):
    x, y = xy
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=lw, edgecolor="black", facecolor="white"
    )
    ax.add_patch(patch)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize)
    return patch

def arrow(ax, start, end, lw=2.2):
    ax.annotate(
        "", xy=end, xytext=start,
        arrowprops=dict(arrowstyle="->", linewidth=lw, color="black", shrinkA=2, shrinkB=2)
    )

def setup_canvas(title, figsize=(16, 9)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title(title, pad=18)
    return fig, ax


## 1. Third-arc position

In [ ]:
fig, ax = setup_canvas("Notebook 67 continues the learning-controller arc", figsize=(16, 9))

# second arc
xs = [0.10, 0.31, 0.52, 0.73]
labels_top = ["43\nResource\nAllocation", "49\nAdaptive\nInfrastructure", "53\nDistributed\nScheduling", "59\nCluster\nOptimization"]
for x, lab in zip(xs, labels_top):
    rounded_box(ax, (x, 0.64), 0.18, 0.13, lab, fontsize=15)
for x in xs[:-1]:
    arrow(ax, (x+0.18, 0.705), (x+0.21, 0.705))
ax.text(0.5, 0.58, "second arc: controller → infrastructure → distributed scheduling → cluster optimization",
        ha="center", va="center", fontsize=17)
ax.plot([0.06, 0.94], [0.53, 0.53], color="black", lw=2)

# bridge arrow
ax.text(0.5, 0.505, "optimized serving traces become training data", ha="center", va="center", fontsize=16)
arrow(ax, (0.82, 0.66), (0.34, 0.43), lw=2.4)

# third arc
xs2 = [0.05, 0.24, 0.43, 0.62, 0.81]
labels_bottom = ["61\nTelemetry\nDataset", "67\nPolicy\nLearning", "71\nOffline\nEvaluation", "73\nSafety\nBounds", "79\nAdaptive\nController"]
for x, lab in zip(xs2, labels_bottom):
    rounded_box(ax, (x, 0.30), 0.17, 0.14, lab, fontsize=15)
for x in xs2[:-1]:
    arrow(ax, (x+0.17, 0.37), (x+0.19, 0.37))

# highlight current notebook
ax.add_patch(FancyBboxPatch((0.24, 0.30), 0.17, 0.14, boxstyle="round,pad=0.035,rounding_size=0.08",
                            linewidth=3.5, edgecolor="black", facecolor="none"))
ax.text(0.5, 0.23, "third arc: telemetry dataset → policy learning → evaluation → safety → controller",
        ha="center", va="center", fontsize=20)

savefig("67_third_arc_position_v2.png")


## 2. Learning pipeline

In [ ]:
fig, ax = setup_canvas("Policy learning specifies adaptive controller candidates", figsize=(16, 9))

labels = [
    "validated\ndataset",
    "state\nwindows",
    "outcome\nlabels",
    "policy\nlearner",
    "candidate\nπθ",
    "offline\nranking",
    "bounded\ncontroller",
]
x0, y, w, h, gap = 0.03, 0.50, 0.125, 0.13, 0.015
for i, lab in enumerate(labels):
    x = x0 + i*(w+gap)
    rounded_box(ax, (x, y), w, h, lab, fontsize=17)
    if i < len(labels)-1:
        arrow(ax, (x+w, y+h/2), (x+w+gap, y+h/2))
rounded_box(ax, (0.18, 0.22), 0.64, 0.12,
            "Notebook 67 learns candidate policies from telemetry records before runtime control.",
            fontsize=18)
savefig("67_learning_pipeline_v2.png")


## 3. Policy model inputs

In [ ]:
fig, ax = setup_canvas("State vector maps telemetry into policy candidates", figsize=(16, 9))

rounded_box(ax, (0.38, 0.45), 0.24, 0.13, "State vector\nS(t)", fontsize=20)
rounded_box(ax, (0.38, 0.29), 0.24, 0.11, "policy model\nπθ(S)", fontsize=18)
rounded_box(ax, (0.38, 0.15), 0.24, 0.10, "action scores\nπθ(a | S)", fontsize=17)
rounded_box(ax, (0.38, 0.03), 0.24, 0.09, "candidate action", fontsize=17)

features = [
    ((0.08, 0.62), "q(t)\nqueue", (0.38, 0.515)),
    ((0.40, 0.72), "u(t)\nutilization", (0.50, 0.58)),
    ((0.70, 0.62), "m(t)\nmemory", (0.62, 0.515)),
    ((0.08, 0.36), "λ(t)\narrival", (0.38, 0.495)),
    ((0.70, 0.36), "v(t)\nverification", (0.62, 0.495)),
]
for xy, text, end in features:
    rounded_box(ax, xy, 0.20, 0.10, text, fontsize=16)
    arrow(ax, (xy[0]+0.10, xy[1]+0.05), end)

rounded_box(ax, (0.40, 0.23), 0.20, 0.08, "r(t)\nreserve", fontsize=15)
arrow(ax, (0.50, 0.31), (0.50, 0.45))
arrow(ax, (0.50, 0.45), (0.50, 0.40))
arrow(ax, (0.50, 0.29), (0.50, 0.25))
arrow(ax, (0.50, 0.15), (0.50, 0.12))

ax.text(0.5, 0.00, r"$S(t)=[q(t), u(t), m(t), \lambda(t), r(t), v(t)]$",
        ha="center", va="bottom", fontsize=24)

savefig("67_policy_model_inputs_v2.png")


## 4. Learning objective

In [ ]:
fig, ax = setup_canvas("Learning objective balances fit, value, and constraints", figsize=(16, 9))

labels = ["prediction\nloss", "value\nloss", "constraint\npenalty", "imbalance\npenalty", "regularized\nobjective"]
x0, y, w, h, gap = 0.12, 0.50, 0.14, 0.11, 0.03
for i, lab in enumerate(labels):
    x = x0 + i*(w+gap)
    rounded_box(ax, (x, y), w, h, lab, fontsize=17)
    if i < len(labels)-1:
        arrow(ax, (x+w, y+h/2), (x+w+gap, y+h/2))

ax.text(0.5, 0.30, r"$\mathcal{L}(\theta)=\ell(\pi_\theta,\pi_0)-\alpha\widehat{U}(\pi_\theta)+\beta C(\pi_\theta)+\gamma R(\pi_\theta)$",
        ha="center", va="center", fontsize=26)
rounded_box(ax, (0.20, 0.13), 0.60, 0.10,
            "The learner is rewarded for useful policy shifts and penalized for unsafe or unstable ones.",
            fontsize=16)
savefig("67_loss_objective_v2.png")


## 5. Validation performance

In [ ]:
epochs = np.arange(1, 41)
train = 0.78*np.exp(-epochs/15) + 0.11
val = 0.70*np.exp(-epochs/16) + 0.23
test = 0.68*np.exp(-epochs/17) + 0.29

plt.figure(figsize=(15, 8))
plt.plot(epochs, train, label="train loss")
plt.plot(epochs, val, label="validation loss")
plt.plot(epochs, test, label="test loss")
plt.axvline(40, linestyle="--", color="black", label="selected epoch 40")
plt.title("Validation performance bounds policy learning")
plt.xlabel("training epoch")
plt.ylabel("loss")
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
savefig("67_generalization_gap_v2.png")


## 6. Counterfactual ranking

In [ ]:
policies = ["baseline", "conservative", "balanced", "aggressive", "unsafe"]
gain = np.array([0.52, 0.57, 0.72, 0.83, 0.92])
latency = -np.array([0.25, 0.21, 0.24, 0.38, 0.62])
spill = -np.array([0.18, 0.14, 0.16, 0.25, 0.48])
net = gain + latency + spill - np.array([0.06, 0.03, 0.05, 0.16, 0.38])

x = np.arange(len(policies))
width = 0.18
plt.figure(figsize=(15, 8))
plt.bar(x - 1.5*width, gain, width, label="gain")
plt.bar(x - 0.5*width, latency, width, label="latency cost")
plt.bar(x + 0.5*width, spill, width, label="spillover cost")
plt.bar(x + 1.5*width, net, width, label="net score")
plt.axhline(0, color="black", lw=1)
plt.xticks(x, policies, rotation=25)
plt.ylabel("offline normalized value")
plt.title("Counterfactual ranking selects bounded policy candidates")
plt.legend(loc="upper left")
savefig("67_counterfactual_ranking_v2.png")


## 7. Constraint regularization

In [ ]:
x = np.linspace(0, 1, 400)
throughput = 1.08*(1 - np.exp(-3.0*x))
latency = 0.18 + 0.75*x**3
constraint_risk = 0.10 + 0.12*np.maximum(0, x-0.65)**2/0.35**2
unregularized = throughput - latency - constraint_risk
regularized = unregularized - 0.50*x**4
best_idx = np.argmax(regularized)
best_x = x[best_idx]

plt.figure(figsize=(15, 8))
plt.plot(x, throughput, label="throughput gain")
plt.plot(x, latency, label="latency pressure")
plt.plot(x, constraint_risk, label="constraint risk")
plt.plot(x, unregularized, label="unregularized objective")
plt.plot(x, regularized, linewidth=3, label="regularized objective")
plt.axvline(best_x, linestyle="--", color="black", label=f"selected ≈ {best_x:.2f}")
plt.title("Constraint regularization avoids unsafe adaptation")
plt.xlabel("policy adaptation intensity")
plt.ylabel("normalized value")
plt.legend(loc="upper left")
plt.grid(alpha=0.25)
savefig("67_constraint_regularization_v2.png")


## 8. Learned policy candidate surface

In [ ]:
reserve = np.linspace(0.05, 0.95, 80)
load = np.linspace(0.05, 0.95, 80)
R, L = np.meshgrid(reserve, load)

# Candidate action categories: 0 steady, 1 reserve, 2 rebalance, 3 scale-out, 4 shed/shorten
score_steady = 0.7*(1-L) + 0.2*(1-R)
score_reserve = 0.6*R + 0.25*(1-L) - 0.35*L
score_rebalance = 0.35 + 0.45*L + 0.20*R - 0.08*np.abs(R-0.55)
score_scale = 0.30 + 0.55*L + 0.30*R - 0.35*(L > 0.93)
score_shed = 0.15 + 0.95*L - 0.75*R
scores = np.stack([score_steady, score_reserve, score_rebalance, score_scale, score_shed], axis=0)
policy = np.argmax(scores, axis=0)

cmap = ListedColormap(plt.cm.viridis(np.linspace(0.05, 0.98, 5)))
norm = BoundaryNorm(np.arange(-0.5, 5.5, 1), cmap.N)

plt.figure(figsize=(13, 9))
im = plt.imshow(policy, origin="lower", extent=[0.05, 0.95, 0.05, 0.95],
                aspect="auto", cmap=cmap, norm=norm, interpolation="nearest")
cbar = plt.colorbar(im, ticks=[0, 1, 2, 3, 4])
cbar.ax.set_yticklabels(["steady", "reserve", "rebalance", "scale-out", "shed/shorten"])
cbar.set_label("selected candidate action")
plt.text(0.18, 0.23, "steady", fontsize=20)
plt.text(0.70, 0.24, "reserve", fontsize=20)
plt.text(0.48, 0.56, "rebalance", fontsize=20)
plt.text(0.73, 0.82, "scale-out", fontsize=20)
plt.text(0.16, 0.75, "shed /\nshorten", fontsize=20, ha="center")
plt.title("Learned policy candidate surface")
plt.xlabel("reserve capacity")
plt.ylabel("active load")
savefig("67_policy_candidate_surface_v2.png")


## 9. Final synthesis

In [ ]:
fig, ax = setup_canvas("Learning dataset specifies policy optimization", figsize=(16, 9))

labels = ["validated\ndataset D", "state-action\nrecords", "policy\nlearner", "candidate\npolicy πθ", "offline\nscore", "controller\ncandidate"]
x0, y, w, h, gap = 0.05, 0.50, 0.14, 0.12, 0.03
for i, lab in enumerate(labels):
    x = x0 + i*(w+gap)
    rounded_box(ax, (x, y), w, h, lab, fontsize=17)
    if i < len(labels)-1:
        arrow(ax, (x+w, y+h/2), (x+w+gap, y+h/2))

rounded_box(ax, (0.18, 0.20), 0.64, 0.11,
            "Learning produces candidates for offline evaluation, not immediate deployment.",
            fontsize=17)
savefig("67_final_synthesis_v2.png")


## Package outputs

In [ ]:
zip_path = Path("notebook_67_figures_v2.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.glob("67_*_v2.png")):
        zf.write(p, arcname=p.name)

manifest = {
    "notebook": "67_policy_learning_v2",
    "repo": "confidence-scheduled-decoding",
    "figures": [p.name for p in sorted(OUT.glob("67_*_v2.png"))],
    "zip": str(zip_path),
}
Path("notebook_67_manifest_v2.json").write_text(json.dumps(manifest, indent=2))
manifest
